<a href="https://colab.research.google.com/github/brunoortega/minicurso_sscad_2026/blob/main/Minicurso_compressao_SSCAD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Verificar visibilidade da GPU

(e instalar NVTOP para visualização do uso da GPU)

In [ ]:
!nvidia-smi

!apt-get install nvtop

# Datasets

### Baixar datasets do repositório **szcompressor**

In [ ]:
!mkdir -p /content/datasets/

%cd /content/datasets/

!echo "downloading 3 fields..."
!wget  https://github.com/szcompressor/cusz-sample-data/raw/main/cesm-CLDHGH-3600x1800.tgz
!wget  https://github.com/szcompressor/cusz-sample-data/raw/main/exafel-59200x388.tgz
!wget  https://github.com/szcompressor/cusz-sample-data/raw/main/hurr-CLOUDf48-500x500x100.tgz

!echo "untaring..."
!tar zxf cesm-CLDHGH-3600x1800.tgz
!tar zxf exafel-59200x388.tgz
!tar zxf hurr-CLOUDf48-500x500x100.tgz

### Baixar datasets do site **SDRBench**

Link: [SDRBench](https://sdrbench.github.io/datasets.html)

In [ ]:
!pip install sdrbench

!mkdir -p /content/datasets/


In [ ]:
!sdrbench download nyx temperature -o /content/datasets/
!sdrbench download hurricane-isabel p -o /content/datasets/

# cuSZp

Repositório: [cuSZp](https://github.com/szcompressor/cuSZp)

Documentação: [Docs](https://cuszp-documentation.readthedocs.io/en/latest/algorithm.html)

## Instalando cuSZp

In [ ]:
%cd /content/
!rm -rf cuSZp
!git clone https://github.com/szcompressor/cuSZp.git

!cd cuSZp && mkdir build

%cd ./cuSZp/build

!cmake -DCMAKE_BUILD_TYPE=Release -DCMAKE_INSTALL_PREFIX=../install/ ..
!make -j
!make install

%cd /content/

#### Teste para ver se está funcionando

In [ ]:
%cd /content/cuSZp/install/bin/

!./cuSZp_test_f32_1D # pode ser os outros testes também
!./cuSZp_test_f32_2D
!./cuSZp_test_f32_3D

%cd /content/

## Compressão em CLI

In [ ]:
%cd /content

!./cuSZp/install/bin/cuSZp -i ./datasets/cesm-CLDHGH-3600x1800 -t f32 -m plain -d 1 -eb rel 1E-2 -x output.cesm_plain_1e-2_d1.cmp -o output.cesm_plain_1e-2_d1.dec
!./cuSZp/install/bin/cuSZp -i ./datasets/cesm-CLDHGH-3600x1800 -t f32 -m fixed -d 1 -eb rel 1E-4 -x output.cesm_fixed_1e-4_d1.cmp -o output.cesm_fixed_1e-4_d1.dec
!./cuSZp/install/bin/cuSZp -i ./datasets/cesm-CLDHGH-3600x1800 -t f32 -m fixed -d 2 1 3600 1800 -eb rel 1E-4 -x output.cesm_fixed_1e-4_d2.cmp -o output.cesm_fixed_1e-4_d2.dec

!./cuSZp/install/bin/cuSZp -i ./datasets/512x512x512/temperature.f32 -t f32 -m plain -d 1 -eb rel 1E-4 -x output.nyx_plain_1e-4_d1.cmp -o output.nyx_plain_1e-4_d1.dec
!./cuSZp/install/bin/cuSZp -i ./datasets/512x512x512/temperature.f32 -t f32 -m plain -d 3 512 512 512 -eb rel 1E-4 -x output.nyx_plain_1e-4_d2.cmp -o output.nyx_plain_1e-4_d2.dec



## Análise da compressão

In [ ]:
from pathlib import Path
import pandas as pd

original = Path("/content/datasets/cesm-CLDHGH-3600x1800")
comprimido = Path("/content/output.cesm_plain_1e-2_d1.cmp")
descomprimido = Path("/content/output.cesm_plain_1e-2_d1.dec")

arquivos = [
    ("Original", original),
    ("Comprimido (cuSZp)", comprimido),
    ("Descomprimido", descomprimido),
]

tamanhos = [(nome, caminho.stat().st_size) for nome, caminho in arquivos]

tamanho_original = tamanhos[0][1]
tamanho_comprimido = tamanhos[1][1]
tamanho_descomprimido = tamanhos[2][1]

df = pd.DataFrame([
    {
        "Arquivo": nome,
        "Tamanho (MB)": round(tamanho / 1e6, 2),
        "Tamanho (MiB)": round(tamanho / 2**20, 2),
        "Tamanho relativo": f"{tamanho / tamanho_original * 100:.2f}%"
    }
    for nome, tamanho in tamanhos
])

display(df)

print(f"Razão de compressão: {tamanho_original / tamanho_comprimido:.2f}x")
print(f"Redução de tamanho: {(1 - tamanho_comprimido / tamanho_original) * 100:.2f}%")
print(f"Espaço economizado: {(tamanho_original - tamanho_comprimido) / 1e6:.2f} MB")
print(f"Integridade do tamanho após descompressão: "
      f"{'OK' if tamanho_original == tamanho_descomprimido else 'TAMANHOS DIFERENTES'}")

## Análise da qualidade da compressão

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from skimage.metrics import structural_similarity

# Arquivos
original = Path("/content/datasets/cesm-CLDHGH-3600x1800")
comprimido = Path("/content/output.cesm_plain_1e-2_d1.cmp")
descomprimido = Path("/content/output.cesm_plain_1e-2_d1.dec")

# Configuração dos dados
dtype = np.float32
shape = (1800, 3600)

# Leitura dos arquivos
dados_originais = np.fromfile(original, dtype=dtype)
dados_reconstruidos = np.fromfile(descomprimido, dtype=dtype)

# Verificação
assert dados_originais.size == np.prod(shape), \
    "Quantidade de elementos do arquivo original incorreta."

assert dados_reconstruidos.size == dados_originais.size, \
    "Quantidade de elementos diferente após descompressão."

# Conversão para matriz
dados_originais = dados_originais.reshape(shape)
dados_reconstruidos = dados_reconstruidos.reshape(shape)

# Conversão para float64 para evitar erros de precisão
ref = dados_originais.astype(np.float64)
rec = dados_reconstruidos.astype(np.float64)

# Erro
erro = ref - rec

# RMSE
rmse = np.sqrt(np.mean(erro ** 2))

# PSNR
data_range = np.max(ref) - np.min(ref)

if rmse == 0:
    psnr = float("inf")
else:
    psnr = 20 * np.log10(data_range / rmse)

# SSIM
ssim = structural_similarity(
    ref,
    rec,
    data_range=data_range,
    gaussian_weights=True,
    sigma=1.5,
    use_sample_covariance=False
)

# Outros indicadores úteis
mae = np.mean(np.abs(erro))
erro_maximo = np.max(np.abs(erro))
erro_relativo = rmse / np.sqrt(np.mean(ref ** 2))

# Resultados
df_qualidade = pd.DataFrame([{
    "RMSE": rmse,
    "MAE": mae,
    "Erro máximo": erro_maximo,
    "PSNR (dB)": psnr,
    "SSIM": ssim,
    "NRMSE": erro_relativo
}])

display(df_qualidade)

In [ ]:
import matplotlib.pyplot as plt

erro = ref - rec

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

im0 = axes[0].imshow(ref, cmap="viridis")
axes[0].set_title("Dados originais")
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(rec, cmap="viridis")
axes[1].set_title("Dados reconstruídos (cuSZp)")
plt.colorbar(im1, ax=axes[1])

im2 = axes[2].imshow(erro, cmap="RdBu_r")
axes[2].set_title("Erro de reconstrução")
plt.colorbar(im2, ax=axes[2])

plt.tight_layout()
plt.show()

## Compressão em Python

[Exemplos com cuSZp em Python](https://github.com/szcompressor/cuSZp/tree/master/python)

In [ ]:
%cd /content/

!pip install pycuda torch numpy


In [ ]:
%cd /content/cuSZp/python/

# This shows cuSZp compresses a HPC field with f32 format.
# Usage: python example-hpc.py <dataset_file> <data_type: f32|f64> <mode: fixed|plain|outlier> <dim: 1|2|3> <dim_z> <dim_y> <dim_x> <error_bound>
#        if dim=1, dim_z, dim_y, and dim_x can be any value (e.g., 0).
#        dim_x is the fastest changing dimension.
# Example: python example-hpc.py data.f32 f32 plain 3 100 100 100 1e-3
#          python example-hpc.py data.f64 f64 outlier 2 0 1000 1000 1e-4
#          python example-hpc.py data.f32 f32 fixed 1 0 0 0 1e-2

!python ./example-hpc.py /content/datasets/cesm-CLDHGH-3600x1800 f32 plain 1 0 0 0 1e-4

# GPUZIP

Repositório: [GPUZIP](https://github.com/LSC-Unicamp/GPUZIP)

## Exemplos de implementação usando C/C++ e CUDA

Acessar o link [GPUZIP C/C++ and CUDA Examples](https://github.com/LSC-Unicamp/GPUZIP/blob/main/docs/CppCudaExamples.md)

## Compressão em Python com GPUZIPy

Acessar o link [GPUZIPy Example](https://colab.research.google.com/github/LSC-Unicamp/GPUZIP/blob/main/GPUZIPy/example/GPUZIPy_Example.ipynb)

## Etc

In [ ]:
# Prebuilt CUDA 12 wheel (Linux x86_64, Python 3.10-3.13): no compilation needed.
!pip install gpuzipy --only-binary gpuzipy --find-links https://github.com/LSC-Unicamp/GPUZIP/releases/expanded_assets/2.1.0


# Alternatively, build from source (~7 min on Colab):
# !git clone https://github.com/LSC-Unicamp/GPUZIP.git && cd GPUZIP/GPUZIPy && pip install .

Constants & Imports

In [ ]:
"""Round-trip compression: compress with cuZFP/Bitcomp, decompress,
and report PSNR, SSIM, and the achieved compression rate."""
import urllib.request
import zipfile
from pathlib import Path
from IPython.display import Image, display

import cupy as cp
import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
from skimage.metrics import structural_similarity

from gpuzipy import (
    CompressorBitcomp,
    CompressorZFP,
    CompressorCuszp,
    compress,
    compressed_buffer_max_size,
    compressed_buffer_size,
    decompress,
)

CUZFP = 0
BITCOMP = 1
CUSZP = 2

DATA_DIR = Path('/data')
MARMOUSI3D_URL = 'https://redu.unicamp.br/api/access/datafile/13547?gbrecs=true'
MARMOUSI3D_ARCHIVE = DATA_DIR / 'marmousi3d.bin.zipfile'
MARMOUSI3D_BIN = DATA_DIR / 'marmousi3d.bin'
MARMOUSI3D_N1, MARMOUSI3D_N2, MARMOUSI3D_N3 = 351, 901, 301
MARMOUSI3D_SHAPE = (MARMOUSI3D_N3, MARMOUSI3D_N2, MARMOUSI3D_N1)

OUTPUT_DIR = Path('./output')

### Downloading the data for tests (Marmousi-3D)

In [ ]:
def download_marmousi3d():
    """Download (if needed) and load the Marmousi3D velocity model as a float32 array."""
    DATA_DIR.mkdir(exist_ok=True)

    if not MARMOUSI3D_ARCHIVE.exists():
        print(f'Downloading {MARMOUSI3D_URL}')
        urllib.request.urlretrieve(MARMOUSI3D_URL, MARMOUSI3D_ARCHIVE)

    if not MARMOUSI3D_BIN.exists():
        with zipfile.ZipFile(MARMOUSI3D_ARCHIVE) as archive:
            archive.extract(MARMOUSI3D_BIN.name, DATA_DIR)

    return np.fromfile(MARMOUSI3D_BIN, dtype=np.float32).reshape(MARMOUSI3D_SHAPE)


data = download_marmousi3d()
print('Loaded Marmousi3D:', data.shape, data.dtype)

Displaying a slide of the downloaded (3D) data

In [ ]:
mid = data.shape[0] // 2
plt.figure(figsize=(10, 4))
plt.gca().invert_yaxis()
plt.imshow(data[mid].T, cmap='gray')
plt.title(f'Marmousi3D — depth slice {mid}')
plt.colorbar(label='velocity')
plt.show()

### Quality Metrics

In [ ]:
def psnr(baseline, comparison):
    '''
    Based on github.com/LSC-Unicamp/SeisUtils
    '''
    mse = np.mean((baseline - comparison) ** 2)
    if mse == 0:
        raise ValueError('Oops, MSE == 0')

    if np.isinf(mse):
        return 0, 0

    max_pixel = np.max(comparison)
    psnr_val = 20 * np.log10(max_pixel / np.sqrt(mse))

    mre = np.mean(np.abs(baseline - comparison)) / np.mean(np.abs(baseline))

    return float(psnr_val), float(mse), float(mre)

def ssim(reference, test):
    """Average structural similarity (SSIM) across depth slices of two 3D arrays."""
    data_range = reference.max() - reference.min()
    scores = [
        structural_similarity(reference[i], test[i], data_range=data_range)
        for i in range(reference.shape[0])
    ]
    return float(np.mean(scores))

### Rendering helpers

In [ ]:
def save_slice_image(frame, path, cmap='gray', colorbar=False, title=None):
    """Save a 2D array as an image."""
    plt.figure()
    plt.gca().invert_yaxis()
    im = plt.imshow(frame, cmap=cmap)
    if colorbar:
        plt.colorbar(im)
    if title:
        plt.title(title)
    plt.savefig(path, bbox_inches='tight', dpi=150)
    plt.close()


def show_comparison(name, original, decompressed, psnr_value, ssim_value):
    mid = original.shape[0] // 2
    orig = original[mid].T
    dec = decompressed[mid].T
    err = np.abs(orig - dec)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax in axes:
        ax.invert_yaxis()

    axes[0].imshow(orig, cmap='gray')
    axes[0].set_title(f'{name} — original')
    axes[1].imshow(dec, cmap='gray')
    axes[1].set_title(f'{name} — decompressed')
    im = axes[2].imshow(err, cmap='viridis')
    axes[2].set_title(f'PSNR {psnr_value:.2f} dB | SSIM {ssim_value:.4f}')
    fig.colorbar(im, ax=axes[2], fraction=0.046)

    plt.tight_layout()
    plt.show()

### The compress/decompress round-trip with GPUZIP

In [ ]:
def test_compression(h_uncompressed, config, name):
    """Compress/decompress per `config`, returning (psnr, ssim, compression_rate)."""
    print('Compressing with', config)
    n1, n2, n3 = h_uncompressed.shape

    compressor = None
    if config['compressor'] == BITCOMP:
        config_kind = 2  # STATIC_DELTA
        compressor = CompressorBitcomp(
            n1, n2, n3, config_kind, 0.0, 0.0, config['delta'], 'float', 'default')
    elif config['compressor'] == CUZFP:
        compressor = CompressorZFP(n1, n2, n3, 'float', config['rate'])
    elif config['compressor'] == CUSZP:
        compressor = CompressorCuszp(n1, n2, n3, config['error_bound'])

    d_uncompressed = cp.asarray(h_uncompressed)

    estimated_size = compressed_buffer_max_size(compressor)
    d_compressed_ptr = cp.cuda.malloc_async(estimated_size)
    d_decompressed = cp.empty((n1, n2, n3), dtype=np.float32)

    actual_size = compress(compressor, d_uncompressed.data.ptr, d_compressed_ptr.ptr)

    # actual_size = compressed_buffer_size(compressor, d_compressed_ptr.ptr)

    decompress(compressor, d_compressed_ptr.ptr, d_decompressed.data.ptr)

    compression_rate = d_decompressed.nbytes / actual_size

    h_decompressed = d_decompressed.get()
    psnr_value, mse, mre = psnr(h_uncompressed, h_decompressed)
    ssim_value = ssim(h_uncompressed, h_decompressed)
    show_comparison(name, h_uncompressed, h_decompressed, psnr_value, ssim_value)

    return psnr_value, ssim_value, compression_rate

### Compressing with cuZFP


In [ ]:
psnr_value, ssim_value, compression_rate = test_compression(
    data, {'rate': 16, 'compressor': CUZFP}, 'cuzfp')
print('cuZFP  → psnr:', psnr_value, 'ssim:', ssim_value, 'rate:', compression_rate)

### Compressing with Bitcomp

In [ ]:
psnr_value, ssim_value, compression_rate = test_compression(
    data, {'delta': 1e-2, 'compressor': BITCOMP}, 'bitcomp')
print('Bitcomp → psnr:', psnr_value, 'ssim:', ssim_value, 'rate:', compression_rate)

### Compressing with cuSZ

In [ ]:
psnr_value, ssim_value, compression_rate = test_compression(
    data, {'error_bound': 1e-4, 'compressor': CUSZP}, 'cuszp')
print('cuszp → psnr:', psnr_value, 'ssim:', ssim_value, 'rate:', compression_rate)